# 07 – Generalización: el mismo modelo con otros datos
**Objetivo:** tomar el mejor modelo ya entrenado (escogido por macro-F1 de **validación**, nunca por prueba) y evaluarlo **sin reentrenar** en otro dataset de yuca. Responde a la pregunta del profesor y a la validez externa que pide el enunciado.

**Fuentes posibles (variable `FUENTE`):**

| Fuente | Clases | Origen | Ventaja | Cuidado |
|---|---|---|---|---|
| `"mendeley"` (Mendeley 2021, DOI 10.17632/3832tx2cb2.1) | 3: sana (91), CBB (49), mosaico CMV → `cmd` (88); 228 fotos | India (Tamil Nadu), 512×512 | Dominio realmente distinto (otro país, otra cámara); descarga sin cuenta | Solo 3 de nuestras 5 clases |
| `"kaggle2020"` (Cassava Leaf Disease Classification) | las mismas 5 | Uganda (Makerere) | Mismas clases y mismo orden de etiquetas | Requiere cuenta de Kaggle; mismo programa de recolección que iCassava 2019, así que puede compartir fotos con nuestro entrenamiento y no es un dominio externo |

Se usa la misma normalización y el mismo tamaño (224×224). Para comparar de forma justa, el modelo también se evalúa en nuestro test de Uganda restringido a las mismas clases.

In [ ]:
# 1) Librerías y reglas del protocolo (iguales en todos los notebooks)
!pip install -q importlib_resources   # sin esto tfds.load falla en Colab

import os, time, json, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

SEED = 42      # semilla fija: así los resultados se pueden repetir
IMG = 224      # tamaño de entrada que esperan MobileNetV2 y ResNet50V2
BATCH = 32     # fotos por lote
EPOCHS = 10    # presupuesto máximo de épocas, igual para todos los experimentos
AUTOTUNE = tf.data.AUTOTUNE
tf.keras.utils.set_random_seed(SEED)

RESULTS = "results"
os.makedirs(RESULTS, exist_ok=True)

# Lo que ya está subido al repositorio (pesos de E1, índices excluidos, etc.) se descarga de GitHub
REPO_RAW = "https://raw.githubusercontent.com/cristianh1818/Proyecto-Yuca-Machine-learning/main/results/"

def traer(archivo):
    """Devuelve la ruta de results/<archivo>. Si no está en Colab, lo baja del repositorio."""
    ruta = f"{RESULTS}/{archivo}"
    if not os.path.exists(ruta):
        r = requests.get(REPO_RAW + archivo, timeout=600)
        r.raise_for_status()
        with open(ruta, "wb") as f:
            f.write(r.content)
    return ruta

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

## Configuración

In [ ]:
FUENTE = "mendeley"        # "mendeley" (por defecto, sin cuenta) o "kaggle2020"
N_KAGGLE = 2000            # fotos de Kaggle a evaluar (muestra estratificada), si FUENTE = "kaggle2020"

# Clases del modelo, en el mismo orden que TFDS
CLASES = ["cbb", "cbsd", "cgm", "cmd", "healthy"]
N_CLASES = len(CLASES)

In [ ]:
# 6) Construcción de modelos congelados (la misma cabeza que E1)
def construir_congelado(Backbone, nombre, aumentar=False):
    """Backbone de ImageNet congelado + cabeza: GAP -> Dropout(0.2) -> Dense(5)."""
    base = Backbone(include_top=False, weights="imagenet", input_shape=(IMG, IMG, 3))
    base.trainable = False                                   # congelado: no se tocan sus pesos
    entradas = tf.keras.Input(shape=(IMG, IMG, 3))
    x = entradas
    if aumentar:
        # Augmentation moderado: estas capas SOLO actúan durante el entrenamiento
        x = tf.keras.layers.RandomFlip("horizontal_and_vertical", seed=SEED)(x)
        x = tf.keras.layers.RandomRotation(0.05, seed=SEED)(x)   # ±18 grados aprox.
    # MobileNetV2 y ResNet50V2 esperan píxeles en [-1, 1] (equivale a su preprocess_input)
    x = tf.keras.layers.Rescaling(1.0 / 127.5, offset=-1)(x)
    x = base(x, training=False)                              # BatchNorm en modo inferencia
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    salidas = tf.keras.layers.Dense(N_CLASES, activation="softmax")(x)
    return tf.keras.Model(entradas, salidas, name=nombre)

def cargar_modelo(nombre):
    """Carga results/<nombre>.keras (o lo baja de GitHub). E3 se puede rearmar desde su cabeza."""
    ruta = f"{RESULTS}/{nombre}.keras"
    if not os.path.exists(ruta):
        try:
            traer(f"{nombre}.keras")
        except Exception:
            pass
    if os.path.exists(ruta):
        return tf.keras.models.load_model(ruta)
    if nombre.startswith("E3"):
        # ResNet50V2 pesa ~95 MB y no cabe en GitHub web: se rearma con ImageNet + la cabeza guardada
        datos = np.load(traer(f"{nombre}_cabeza.npz"))
        m = construir_congelado(tf.keras.applications.ResNet50V2, nombre)
        m.layers[-1].set_weights([datos["W"], datos["b"]])
        return m
    raise FileNotFoundError(f"No se encontró {nombre}.keras ni en Colab ni en el repositorio")

## Escoger el mejor modelo (por macro-F1 de validación)

In [ ]:
CANDIDATOS = ["E1_mobilenetv2_congelado", "E2_mobilenetv2_finetuning",
              "E3_resnet50v2_congelado", "E1_augmentation"]
puntajes = {}
for nombre in CANDIDATOS:
    try:
        r = json.load(open(traer(f"{nombre}.json")))
        puntajes[nombre] = r["validacion"]["macro_f1"]
    except Exception:
        print("Todavía no está:", nombre)
print(puntajes)
MEJOR = max(puntajes, key=puntajes.get)     # se puede cambiar a mano, p. ej. MEJOR = "E1_mobilenetv2_congelado"
print("Mejor modelo por macro-F1 de validación:", MEJOR)
modelo = cargar_modelo(MEJOR)

## Descargar el otro dataset

In [ ]:
import glob, zipfile, subprocess
DEST = f"otros_datos/{FUENTE}"
os.makedirs(DEST, exist_ok=True)
rutas, etiquetas = [], []

if FUENTE == "mendeley":
    # API pública de Mendeley Data (no requiere cuenta). El dataset tiene 3 carpetas:
    # "Cassava CB (Cassava Blight)", "Cassava CM (Cassava Mosaic)" y "Cassava Healthy leaf"
    BASE_API = "https://data.mendeley.com/public-api/datasets/3832tx2cb2"
    def clase_de_carpeta(nombre):
        n = nombre.lower()
        if "blight" in n: return "cbb"       # tizón bacteriano
        if "mosaic" in n: return "cmd"       # virus del mosaico (CMV) = nuestra clase cmd
        if "healthy" in n: return "healthy"
        return None
    for carpeta in requests.get(f"{BASE_API}/folders/1", timeout=60).json():
        c = clase_de_carpeta(carpeta["name"])
        if c is None:
            continue
        os.makedirs(f"{DEST}/{c}", exist_ok=True)
        archivos = requests.get(f"{BASE_API}/files?folder_id={carpeta['id']}&version=1", timeout=60).json()
        print(f"{carpeta['name']} -> {c}: {len(archivos)} fotos")
        for f in archivos:
            ruta = f"{DEST}/{c}/{f['filename']}"
            if not os.path.exists(ruta):
                r = requests.get(f["content_details"]["download_url"], timeout=120)
                with open(ruta, "wb") as g:
                    g.write(r.content)
            rutas.append(ruta); etiquetas.append(CLASES.index(c))

elif FUENTE == "kaggle2020":
    # Requiere kaggle.json (kaggle.com → Settings → Create New Token) y aceptar las reglas de la competencia
    if not os.path.exists("/root/.kaggle/kaggle.json"):
        from google.colab import files
        print("Sube tu kaggle.json"); files.upload()
        os.makedirs("/root/.kaggle", exist_ok=True)
        os.replace("kaggle.json", "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
    subprocess.run(["pip", "install", "-q", "kaggle"], check=True)
    subprocess.run(["kaggle", "competitions", "download", "-c", "cassava-leaf-disease-classification",
                    "-p", DEST], check=True)
    zf = zipfile.ZipFile(f"{DEST}/cassava-leaf-disease-classification.zip")
    zf.extract("train.csv", DEST)
    tabla_k = pd.read_csv(f"{DEST}/train.csv")
    # En Kaggle 2020: 0=CBB, 1=CBSD, 2=CGM, 3=CMD, 4=Healthy -> mismo orden que nuestras clases
    muestra = (tabla_k.groupby("label", group_keys=False)
                      .apply(lambda g: g.sample(frac=N_KAGGLE / len(tabla_k), random_state=SEED)))
    for _, fila in muestra.iterrows():
        nombre_zip = f"train_images/{fila['image_id']}"
        zf.extract(nombre_zip, DEST)
        rutas.append(f"{DEST}/{nombre_zip}"); etiquetas.append(int(fila["label"]))

y_otro = np.array(etiquetas)
PRESENTES = sorted(set(etiquetas))
print("Fotos por clase:", {CLASES[c]: int((y_otro == c).sum()) for c in PRESENTES})

## Mismo preprocesamiento: 224×224 y la normalización que lleva el modelo adentro

In [ ]:
def leer(p):
    img = tf.image.decode_image(tf.io.read_file(p), channels=3, expand_animations=False)
    return tf.cast(tf.cast(tf.image.resize(img, (IMG, IMG)), tf.uint8), tf.float32)

ds_otro = (tf.data.Dataset.from_tensor_slices(rutas)
           .map(leer, num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE))

# Nuestro test de Uganda (mismas fotos filtradas), para comparar en las mismas clases
ds_test = tfds.load("cassava", split="test", as_supervised=True)
excl = json.load(open(traer("indices_excluidos_final.json")))["test"]
quitar = tf.constant(sorted(excl), dtype=tf.int64)
ds_test = ds_test.enumerate().filter(
    lambda i, xy: tf.logical_not(tf.reduce_any(tf.equal(i, quitar)))).map(lambda i, xy: xy)
test = (ds_test.map(lambda x, y: (tf.cast(tf.cast(tf.image.resize(x, (IMG, IMG)), tf.uint8), tf.float32), y))
        .batch(BATCH).prefetch(AUTOTUNE))
y_test = np.concatenate([y.numpy() for _, y in test])

## Evaluar sin reentrenar

In [ ]:
AUSENTES = [c for c in range(N_CLASES) if c not in PRESENTES]

def metricas(probs, y, etiqueta):
    pred = probs.argmax(axis=1)
    pred_restr = np.array(PRESENTES)[probs[:, PRESENTES].argmax(axis=1)]   # solo entre las clases presentes
    rep = classification_report(y, pred, labels=PRESENTES, target_names=[CLASES[c] for c in PRESENTES],
                                output_dict=True, zero_division=0)
    return {"datos": etiqueta, "n": int(len(y)),
            "accuracy": round(accuracy_score(y, pred), 4),
            "macro_f1": round(f1_score(y, pred, labels=PRESENTES, average="macro", zero_division=0), 4),
            "macro_f1_restringido": round(f1_score(y, pred_restr, average="macro", zero_division=0), 4),
            "f1_por_clase": {CLASES[c]: round(rep[CLASES[c]]["f1-score"], 4) for c in PRESENTES},
            "recall_por_clase": {CLASES[c]: round(rep[CLASES[c]]["recall"], 4) for c in PRESENTES},
            "pct_predice_clase_ausente": round(100 * np.isin(pred, AUSENTES).mean(), 2)}, pred

t0 = time.time()
probs_otro = modelo.predict(ds_otro, verbose=0)
ms_img = 1000 * (time.time() - t0) / len(y_otro)

mask = np.isin(y_test, PRESENTES)
probs_uga = modelo.predict(test, verbose=0)[mask]

m_uga, _ = metricas(probs_uga, y_test[mask], "Uganda (test propio, mismas clases)")
m_otro, pred_otro = metricas(probs_otro, y_otro, f"{FUENTE} (sin reentrenar)")
m_otro["inferencia_ms_por_imagen"] = round(ms_img, 3)

resultado = {"modelo": MEJOR, "fuente": FUENTE, "clases_presentes": [CLASES[c] for c in PRESENTES],
             "uganda": m_uga, "otro": m_otro}
with open(f"{RESULTS}/generalizacion_{FUENTE}.json", "w") as f:
    json.dump(resultado, f, indent=2)
np.save(f"{RESULTS}/generalizacion_{FUENTE}_probs.npy", probs_otro)
pd.DataFrame([m_uga, m_otro]).drop(columns=["f1_por_clase", "recall_por_clase"])

In [ ]:
print(json.dumps(resultado, indent=2))
# Matriz de confusión: filas = clases reales presentes, columnas = las 5 clases del modelo
cm = confusion_matrix(y_otro, pred_otro, labels=range(N_CLASES))[PRESENTES]
fig, ax = plt.subplots(figsize=(7, 1.2 + 0.8 * len(PRESENTES)))
ax.imshow(cm, cmap="Greens")
ax.set_xticks(range(N_CLASES)); ax.set_xticklabels(CLASES)
ax.set_yticks(range(len(PRESENTES))); ax.set_yticklabels([CLASES[c] for c in PRESENTES])
for i in range(len(PRESENTES)):
    for j in range(N_CLASES):
        ax.text(j, i, cm[i, j], ha="center", va="center")
ax.set_xlabel("predicción"); ax.set_ylabel("real")
ax.set_title(f"{MEJOR} evaluado en {FUENTE} (sin reentrenar)")
plt.tight_layout(); plt.savefig(f"{RESULTS}/generalizacion_{FUENTE}_matriz.png", dpi=120); plt.show()

## Ejemplos de errores en el otro dataset

In [ ]:
errores = np.where(pred_otro != y_otro)[0]
muestra = np.random.default_rng(SEED).choice(errores, size=min(8, len(errores)), replace=False)
fig, axs = plt.subplots(2, 4, figsize=(14, 7))
for ax, i in zip(axs.ravel(), muestra):
    ax.imshow(leer(rutas[i]).numpy().astype("uint8")); ax.axis("off")
    ax.set_title(f"real: {CLASES[y_otro[i]]} | pred: {CLASES[pred_otro[i]]}\nconfianza: {probs_otro[i].max():.2f}",
                 fontsize=9)
plt.suptitle(f"{MEJOR} en {FUENTE}: errores"); plt.tight_layout()
plt.savefig(f"{RESULTS}/generalizacion_{FUENTE}_errores.png", dpi=120); plt.show()

## Tabla consolidada (E0–E3, adicional y generalización)
Lee los JSON de `results/` (o del repositorio) y genera `tabla_consolidada.csv` y `.md`.

In [ ]:
EXPERIMENTOS = {"E0": "E0_baseline", "E1": "E1_mobilenetv2_congelado", "E2": "E2_mobilenetv2_finetuning",
                "E3": "E3_resnet50v2_congelado", "Extra (E1+aug)": "E1_augmentation"}
filas = []
for corto, nombre in EXPERIMENTOS.items():
    try:
        r = json.load(open(traer(f"{nombre}.json")))
    except Exception:
        print("Falta", nombre); continue
    fila = {"experimento": corto, "acc_val": r["validacion"]["accuracy"], "macroF1_val": r["validacion"]["macro_f1"],
            "acc_test": r["prueba"]["accuracy"], "macroF1_test": r["prueba"]["macro_f1"]}
    fila.update({f"F1_{c}": r["validacion"]["f1_por_clase"][c] for c in CLASES})
    fila.update({f"recall_{c}": r["validacion"]["recall_por_clase"][c] for c in CLASES})
    fila.update({"epocas": r["config"]["epochs_ejecutadas"], "entrenamiento_min": r["tiempo_entrenamiento_min"],
                 "inferencia_ms_test": r["prueba"]["inferencia_ms_por_imagen"], "parametros": r["parametros"],
                 "entrenables": r.get("parametros_entrenables"), "tamano_mb": r["tamano_mb"]})
    filas.append(fila)
filas.append({"experimento": f"Generalización ({MEJOR} en {FUENTE})", "acc_test": m_otro["accuracy"],
              "macroF1_test": m_otro["macro_f1"],
              **{f"F1_{c}": m_otro["f1_por_clase"].get(c) for c in CLASES},
              **{f"recall_{c}": m_otro["recall_por_clase"].get(c) for c in CLASES},
              "inferencia_ms_test": m_otro["inferencia_ms_por_imagen"]})
tabla = pd.DataFrame(filas)
tabla.to_csv(f"{RESULTS}/tabla_consolidada.csv", index=False)
with open(f"{RESULTS}/tabla_consolidada.md", "w") as f:
    try:
        f.write(tabla.to_markdown(index=False))
    except ImportError:            # to_markdown necesita la librería tabulate
        f.write(tabla.to_string(index=False))
pd.set_option("display.max_columns", None)
tabla

## Análisis
[COMPLETAR TRAS EJECUTAR] — ver `analisis_generalizacion.md`.

In [ ]:
# Descargar los resultados para subirlos a la carpeta results/ del repositorio.
# Los .keras grandes (más de 25 MB) no se pueden subir por la web de GitHub: se excluyen.
import glob, zipfile
with zipfile.ZipFile("results_generalizacion.zip", "w") as z:
    for f in glob.glob(f"{RESULTS}/*"):
        if f.endswith(".keras") and os.path.getsize(f) > 25e6:
            print("Se omite por tamaño:", f)
            continue
        z.write(f)
from google.colab import files
files.download("results_generalizacion.zip")